# 08 - Phase 3: why did landmarks hurt?  (landmark ablation)
**Question.** In the main study the full-SoV prompt (V5) and landmarks-only (V7) did *worse* than the plain image, which is the
opposite of the paper. The model answered "neutral" for 82% / 70% of faces. **Hypothesis:** the big landmark dots cover the eyes and mouth
of small faces. **Test:** draw the same landmarks with much smaller dots (V8, V9) and check whether accuracy and the share of "neutral"
answers recover.

| | |
|---|---|
| **Input** | `data/dataset.json` (needs the photos) and the main run in `results/<model>/raw_outputs.jsonl` |
| **Output** | new answers appended to `raw_outputs.jsonl`, `results/<model>/landmark_ablation.csv` |
| **Needs** | GPU (Colab T4) - only the 2 new variants are run, finished answers are skipped |

In [ ]:
# Setup: find the repo root so `from src import ...` works (run this notebook from inside the repo).
import sys
from pathlib import Path
ROOT = next((p for p in [Path.cwd(), Path.cwd().parent] if (p / "src").is_dir()), None)
assert ROOT is not None, "Run from inside the repo (the folder containing src/). In Colab: upload the repo zip, unzip it and %cd into it."
sys.path.insert(0, str(ROOT))
from src import config, utils
print("repo root:", ROOT)

In [ ]:
# Colab only
!pip -q install -U transformers accelerate bitsandbytes pillow tqdm

In [ ]:
PROVIDER = "qwen"
from src import models
DATA = utils.load_json(config.DATASET_JSON)
models.run_experiment(DATA, PROVIDER, config.EXT_VARIANTS)     # only V8 and V9 are new

In [ ]:
import pandas as pd
from src import evaluation
VARS = ["V0_plain", "V4_boxes_ids", "V5_sov_full", "V8_sov_small_dots", "V7_landmarks_only", "V9_landmarks_small_dots"]
RES = config.results_dir(PROVIDER)
RAW = utils.load_raw_outputs(RES / "raw_outputs.jsonl")
res = evaluation.evaluate(DATA, RAW, VARS)
allm = res["M"][res["M"].difficulty == "All"].set_index("variant")
neutral = {v: share for v, top, share in res["top_answers"] if top == "neutral"}
top = {v: (t, s) for v, t, s in res["top_answers"]}
rows = []
for v in VARS:
    rows.append({"Variant": config.VARIANT_TITLES[v], "Accuracy %": allm.loc[v, "accuracy"] * 100, "Macro F1 %": allm.loc[v, "f1"] * 100,
                 "Most common answer": top[v][0], "Its share %": top[v][1] * 100,
                 "Happy recall %": res["per_class"].loc[v, "happy"] * 100 if "happy" in res["per_class"].columns else float("nan")})
TABLE = pd.DataFrame(rows)
pd.options.display.float_format = "{:.1f}".format
print(TABLE.to_string(index=False))
print(f"\nMajority-class baseline: always '{res['maj_class']}' = {res['maj']:.1%}")
print("\nAccuracy difference vs plain, percentage points (95% bootstrap CI):")
CI = res["ci"].copy(); CI["variant"] = CI.variant.map(config.VARIANT_TITLES)
CI[["acc_diff", "ci_low", "ci_high"]] *= 100                       # percentage points
print(CI.rename(columns={"acc_diff": "diff (pts)", "ci_low": "CI low", "ci_high": "CI high"}).to_string(index=False))
TABLE.to_csv(RES / "landmark_ablation.csv", index=False)

## How to read the result (write your own conclusion)
- If V8 / V9 **recover** (higher accuracy than V5 / V7, lower share of "neutral", happy recall back near the plain image) -> the big dots were hiding the faces.
- If they **do not** recover -> dot size is not the cause; the model simply does not use landmark dots (see notebook 09 for a second model).
Either way, report it with the confidence intervals - with few images most differences are not significant.